In [1]:
import pandas as pd
df=pd.read_csv('/home/honglinbao/chain_of_hints/capsule/run/run.csv')
df

,index,A,B,Sequence,Length,TransitPPL,TransitNLL,OutsidePPL,OutsideNLL,Full_Length,Gini
0,0,1543300511,1971883911,"1543300511, 1973834188, 1971883911",2,13.451600,2.599098,17.632233,2.869729,0.691552,0.037951
1,1,1543300511,1971883911,"1543300511, 1964829868, 2870577106, 1971883911",3,9.828362,2.285272,9.015227,2.198915,1.049039,0.143193
2,2,1543300511,1971883911,"1543300511, 2582974950, 1971883911",2,12.079294,2.491493,14.125843,2.648006,0.824174,0.095029
3,3,1543300511,1971883911,"1543300511, 1993663650, 2831405304, 1971883911",3,14.213931,2.654223,16.290224,2.790565,0.825472,0.314420
4,4,1543300511,1971883911,"1543300511, 2170046948, 2064928476, 2511807583...",8,8.628332,2.155051,7.982647,2.077270,2.097455,0.234229
...,...,...,...,...,...,...,...,...,...,...,...
284995,1924495,772101138,2361645016,"772101138, 2299637917, 2785077202, 2361645016",3,22.085677,3.094929,19.919430,2.991696,0.534447,0.038480
284996,1924496,772101138,2361645016,"772101138, 2840282395, 2361645016",2,11.913504,2.477673,8.490015,2.138891,0.386756,0.055154
284997,1924497,772101138,2361645016,"772101138, 2269883212, 3124438215, 2402421039,...",7,6.688183,1.900342,5.873941,1.770526,1.676658,0.204472
284998,1924498,772101138,2361645016,"772101138, 241749838, 1602904958, 2304733506, ...",7,10.828465,2.382178,10.472836,2.348785,1.347186,0.237481


In [6]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm
df = pd.read_csv('/home/honglinbao/chain_of_hints/capsule/run/run.csv')
df['AB_pair'] = df.apply(lambda row: tuple(sorted([row['A'], row['B']])), axis=1)
unique_pairs = df['AB_pair'].unique()
cols = ['OutsideNLL', 'Gini', 'Length', 'TransitNLL']

all_results = []
for pair in tqdm(unique_pairs, desc="Processing AB pairs"):
    pair_df = df[df['AB_pair'] == pair].reset_index(drop=True)

    if len(pair_df) < 10:
        continue  

    for n in range(10, min(801, len(pair_df) + 1)):
        sample = pair_df.sample(n=n, random_state=None).copy()

        scaler = MinMaxScaler()
        scaled = scaler.fit_transform(sample[cols])
        scaled_df = pd.DataFrame(scaled, columns=cols)

        score = (
            scaled_df['OutsideNLL']
            - scaled_df['Gini']
            - scaled_df['Length']
            - scaled_df['TransitNLL']
        )

        max_idx = score.idxmax()
        max_val = score[max_idx]
        max_sequence = sample.iloc[max_idx]['Sequence']

        all_results.append({
            'A': pair[0],
            'B': pair[1],
            'Sample_Size': n,
            'Max_Value': max_val,
            'Sequence': max_sequence
        })

result_df = pd.DataFrame(all_results)
result_df 

Processing AB pairs: 100%|██████████| 190/190 [13:24<00:00,  4.24s/it]


,A,B,Sample_Size,Max_Value,Sequence
0,1543300511,1971883911,10,0.011436,"1543300511, 1548971626, 1971883911"
1,1543300511,1971883911,11,0.000000,"1543300511, 2327410228, 1971883911"
2,1543300511,1971883911,12,-0.146915,"1543300511, 2509961745, 2006049835, 1971883911"
3,1543300511,1971883911,13,-0.004494,"1543300511, 2382278470, 2086159189, 1971883911"
4,1543300511,1971883911,14,-0.201555,"1543300511, 343696830, 1533251519, 2561290740,..."
...,...,...,...,...,...
150285,772101138,2361645016,796,-0.076287,"772101138, 2299485166, 2361645016"
150286,772101138,2361645016,797,-0.025398,"772101138, 2398873331, 2361645016"
150287,772101138,2361645016,798,-0.040396,"772101138, 2299485166, 2361645016"
150288,772101138,2361645016,799,-0.030558,"772101138, 2398873331, 2361645016"


In [15]:
result_df=pd.read_csv('max.csv')
result_df
result_df = result_df.sort_values(by=['A', 'B', 'Sample_Size'])
result_df['Max_Value_Updated'] = result_df.groupby(['A', 'B'])['Max_Value'].cummax()
first_max_rows = result_df.groupby(['A', 'B', 'Max_Value_Updated']).head(1)
cols_to_fill = ['Sequence']  
result_df[cols_to_fill] = (
    result_df.groupby(['A', 'B'])[cols_to_fill].ffill()
)
result_df = result_df.groupby(['A', 'B'], as_index=False).tail(1)
result_df.to_csv('max2.csv')

In [1]:
import pandas as pd
df = pd.read_csv('max2.csv')
all_df = pd.read_parquet('/net/scratch/honglinbao/siyang/coh_2015.parquet')
df['A'] = df['A'].astype(str).str.strip()
df['B'] = df['B'].astype(str).str.strip()
df['Sequence'] = df['Sequence'].astype(str).str.strip()
all_df['id'] = all_df['id'].astype(str).str.strip()
id_to_abs = all_df.set_index('id')['recovered_abstract'].to_dict()
def map_sequence_to_abs(seq):
    ids = [s.strip() for s in seq.split(',')]
    ids = ids[1:-1]
    return ';\n'.join(
        f"intermediate idea {idx+1}: {id_to_abs.get(i, '')}" 
        for idx, i in enumerate(ids)
    )

df['Sequence_abs'] = df['Sequence'].apply(map_sequence_to_abs)
df['A_abs'] = df['A'].map(id_to_abs).fillna('')
df['B_abs'] = df['B'].map(id_to_abs).fillna('')
df

,Unnamed: 0.1,Unnamed: 0,index,A,B,Sequence,Max_Value_Updated,Sequence_abs,A_abs,B_abs
0,0,0,60906,40845802,2518600061,"2518600061, 2118232162, 40845802",0.108516,intermediate idea 1: the $b$-mode of polarized...,the exploitation of the potentials of the upco...,the emission of electromagnetic waves in the r...
1,1,1,93337,309307303,856986474,"856986474, 2588603112, 309307303",0.224771,intermediate idea 1: abstract in my thesis pro...,the swedish forest industry has put a lot of r...,purpose: we propose to apply a probabilistic f...
2,2,2,125768,309307303,2303481816,"2303481816, 2209679201, 309307303",0.144474,intermediate idea 1: previous studies on the a...,the swedish forest industry has put a lot of r...,the present invention discloses a lucid ganode...
3,3,3,52205,309307303,2518600061,"2518600061, 1846150569, 309307303",0.072527,intermediate idea 1: the ads/cft correspondenc...,the swedish forest industry has put a lot of r...,the emission of electromagnetic waves in the r...
4,4,4,147916,772101138,1539304985,"772101138, 1272275799, 1539304985",0.314917,intermediate idea 1: new mobile hotspots are b...,this chapter extends probabilistic analytical ...,this paper presents an overview of the design ...
...,...,...,...,...,...,...,...,...,...,...
185,185,185,117858,2846204682,3141093539,"2846204682, 2757006384, 3141093539",0.491675,intermediate idea 1: rare complication: tapia’...,the present invention relates to a natural vag...,"the characteristics of adsorption, desorption,..."
186,186,186,14237,2862586448,2873126883,"2862586448, 122649270, 2873126883",0.400153,"intermediate idea 1: the lt3756 controller, a ...",the invention discloses an extraction process ...,problem to be solved: to provide an inorganic ...
187,187,187,18983,2862586448,2875911214,"2862586448, 2511918497, 2875911214",0.161494,intermediate idea 1: espanolla reapertura de j...,the invention discloses an extraction process ...,an efficient and improved traffic information ...
188,188,188,18192,2862586448,2881258957,"2862586448, 2902781640, 2881258957",0.288072,intermediate idea 1: this study aims to conduc...,the invention discloses an extraction process ...,the invention provides an implantable multi-ch...


In [2]:
import pandas as pd

df = pd.read_csv('max2.csv')

cols = ['gen_abs', 'cot', 'got', 'tot', 'raw', 'one_shot', 'few_shot']

for col in cols:
    df[col] = (
        df[col]
        .astype('string')          
        .fillna('')                
        .str.strip()
        .str.lower()                
        .str.replace(r'(?s)^.*abstract:\s*', '', regex=True)  
        .str.replace(r'\s+', ' ', regex=True)  
        .str.strip()
    )

df.to_csv('max2.csv', index=False)

In [4]:
import pandas as pd
df = pd.read_csv('max2.csv')
df

,index,A,B,Sequence,Max_Value_Updated,Sequence_abs,A_abs,B_abs,gen_abs
0,7118,1543300511,1683393507,"1543300511, 3018047130, 1683393507",0.290385,intermediate idea 1: explore & learn about the...,poster presentation background asthma is one o...,this paper proposes a method for voltage dip f...,this interdisciplinary study combines the fiel...
1,78308,1922796052,2260058608,"2260058608, 2015177237, 1922796052",0.127044,intermediate idea 1: segment routing enabling ...,the paper proposes a new type of the synchrono...,state immunity is an important public internat...,this paper presents an innovative approach to ...
2,113112,2043421353,2846204682,"2846204682, 2358253076, 2043421353",0.170327,intermediate idea 1: data centre networks are ...,complex indoor and outdoor missions for autono...,the present invention relates to a natural vag...,"in the growing field of autonomous systems, en..."
3,53787,2518600061,2550695374,"2518600061, 2043626635, 2550695374",0.349440,intermediate idea 1: background various studie...,the emission of electromagnetic waves in the r...,users may consume and/or share information thr...,this paper integrates the concepts of electrom...
4,38758,1587045618,1989800096,"1989800096, 2783647754, 1587045618",0.108586,intermediate idea 1: the world population is i...,in this paper we present a channel measurement...,social curation is the process through which t...,as the world population ages and the prevalenc...
...,...,...,...,...,...,...,...,...,...
95,13446,912934696,2862586448,"2862586448, 2108025254, 912934696",0.171843,intermediate idea 1: the therapist-provided co...,with the upcoming transition from petascale to...,the invention discloses an extraction process ...,as high-performance computing (hpc) systems ap...
96,15819,2183625092,2862586448,"2862586448, 2238596166, 2183625092",0.267226,intermediate idea 1: (proquest: ... denotes fo...,large-scale applications expressed as scientic...,the invention discloses an extraction process ...,we propose a novel approach to optimize the ex...
97,9491,1543300511,2218851917,"1543300511, 2725032403, 2218851917",0.123878,"intermediate idea 1: abstrak suhardi, a. 2015....",poster presentation background asthma is one o...,the building energy model(bem) has been widely...,this study aims to develop a novel approach to...
98,93337,309307303,856986474,"856986474, 2588603112, 309307303",0.224771,intermediate idea 1: abstract in my thesis pro...,the swedish forest industry has put a lot of r...,purpose: we propose to apply a probabilistic f...,developing robust communication protocols in c...
